# 03 — Optimisation & Parametric Analysis (Task 3)

**Goals (Milestone 3):**

1. Sweep **search-depth cutoffs** and measure win-rate, draw-rate, and time-per-move.
2. Sweep **heuristic weights** to find a configuration that beats the default.
3. Sweep **n_prealloc ∈ {1, 2, 3}** to characterise the effect of randomisation.
4. Cache every experiment to `cache/results/` so re-runs are cheap.

All cached results are pickled CSV files; deleting `cache/` forces a recompute.

In [1]:
# Standard imports plus the executor we use to spread games across CPU cores.
import os, sys, random, time, itertools, pickle, atexit
from concurrent.futures import ProcessPoolExecutor
import pandas as pd

# Notebooks live in ./notebooks/, so add the project root to sys.path
# before importing our local modules.
_here = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _here not in sys.path:
    sys.path.insert(0, _here)
import settings, tictactoe66

game = tictactoe66.TicTacToe66()
MAX_WORKERS = os.cpu_count() or 1

# Spawning worker processes on Windows is expensive (each one re-imports
# pandas, AIMA, etc.).  Allocate the pool once at notebook startup and
# reuse it across every experiment instead of paying that cost per call.
POOL = ProcessPoolExecutor(max_workers=MAX_WORKERS)
atexit.register(POOL.shutdown, wait=False, cancel_futures=True)

print("cache dir:", settings.CACHE_DIR)
print("workers :", MAX_WORKERS)

cache dir: C:\Git\AI801Project\cache
workers : 32


## Helpers

Each experiment below boils down to "play a lot of games and tabulate the results".
We feed every game to a single shared `ProcessPoolExecutor` so all CPU cores stay
busy and we only pay the (slow) Windows worker-spawn cost once.

Sides are alternated game-by-game (`a_is_X = i % 2 == 0`) so the X-first advantage
cancels out, and each game gets its own deterministic seed so re-runs are reproducible
regardless of the order in which workers finish.

In [2]:
def run_or_load(name, fn, force=None):
    """Memoise an experiment to ``cache/results/<name>.pkl``.

    ``force`` defaults to ``settings.FORCE_RECOMPUTE`` so the recompute
    switch lives in one place (``settings.py``) rather than at each call site.
    """
    if force is None:
        force = settings.FORCE_RECOMPUTE
    p = settings.cache_path(name + ".pkl")
    if p.exists() and not force:
        print(f"[cache] loading {p.name}")
        return pickle.loads(p.read_bytes())
    print(f"[run]   computing {p.name} ...")
    t0 = time.perf_counter()
    out = fn()
    print(f"        done in {time.perf_counter() - t0:.1f}s")
    p.write_bytes(pickle.dumps(out))
    return out


def _spec(player_kind, depth=None, weights=None):
    """Encode a player as something the worker can pickle and rebuild."""
    if player_kind == "random":
        return "random"
    return (depth, weights)


def make_balanced_jobs(spec_a, spec_b, n_prealloc, n_games, seed, tag):
    """Build n_games worker jobs with sides alternated each game.

    All jobs share the same ``tag`` so we can split results back out by
    configuration after the batch returns.
    """
    jobs = []
    for i in range(n_games):
        a_is_X = (i % 2 == 0)
        per_seed = seed * 1_000_003 + i  # stable hash-ish seed per game
        jobs.append((spec_a, spec_b, n_prealloc, per_seed, a_is_X, tag))
    return jobs


def run_games(jobs):
    """Submit a flat list of jobs to the shared pool and return results.

    Submitting one big batch (rather than one batch per configuration) is
    what keeps every core busy: short games no longer have to wait for the
    longest game in their batch before the next batch can start.
    """
    tags = [j[5] for j in jobs]
    # The worker signature wants game_idx in slot 5; replace our tag with
    # a unique per-job index and stash the original tag back on afterwards.
    worker_jobs = [j[:5] + (i,) for i, j in enumerate(jobs)]
    results = list(POOL.map(tictactoe66._play_one_worker, worker_jobs, chunksize=1))
    for r, tag in zip(results, tags):
        r["tag"] = tag
    return results


def games_to_df(results, label_a="A", label_b="B"):
    """Turn worker results into a DataFrame with human-friendly labels."""
    df = pd.DataFrame(results)
    df["winner_label"] = df["winner_label"].map(
        {"A": label_a, "B": label_b, "draw": "draw"})
    return df

## Experiment A -- depth sweep, AB(d) vs AB(d=1)

Random-move opponents are too weak to discriminate between depths (even d=1
with our heuristic beats them every time), so we use AB(d=1) as the baseline.
Deeper search should win more games and -- since it sees mating sequences
earlier -- finish them in fewer plies.

In [3]:
def expA():
    baseline = _spec("ab", depth=1)
    jobs = []
    for d in settings.DEPTH_SWEEP:
        deep = _spec("ab", depth=d)
        jobs.extend(make_balanced_jobs(
            deep, baseline, n_prealloc=2,
            n_games=settings.N_GAMES_PER_CONFIG,
            seed=100 + d, tag=d))
    df = games_to_df(run_games(jobs), label_a="deep", label_b="d1")
    return df.rename(columns={"tag": "depth"})

dfA = run_or_load("expA_depth_vs_d1", expA)
summary_A = (dfA.groupby("depth")
                .agg(win_deep =("winner_label", lambda s: (s == "deep").mean()),
                     loss_deep=("winner_label", lambda s: (s == "d1").mean()),
                     draw     =("winner_label", lambda s: (s == "draw").mean()),
                     mean_plies  =("plies",   "mean"),
                     mean_seconds=("seconds", "mean")))
summary_A

[run]   computing expA_depth_vs_d1.pkl ...
        done in 20.7s


,win_deep,loss_deep,draw,mean_plies,mean_seconds
depth,,,,,
1,0.5125,0.4125,0.075,11.7375,0.080330
2,0.4625,0.5125,0.025,11.1750,0.479575
3,0.5875,0.3625,0.050,13.1750,4.588749


## Experiment B -- pre-allocation sweep

How does the number of randomly placed initial markers affect the outcome
distribution?  We play AB(d=DEFAULT) against itself with sides alternated;
the proposal predicts that more pre-allocation should mean fewer ties and
shorter games.

In [4]:
def expB():
    ab = _spec("ab", depth=settings.DEFAULT_DEPTH)
    jobs = []
    for n in settings.PREALLOC_CHOICES:
        jobs.extend(make_balanced_jobs(
            ab, ab, n_prealloc=n,
            n_games=settings.N_GAMES_PER_CONFIG,
            seed=200 + n, tag=n))
    # play_one_game already records n_prealloc in each result, so we just
    # drop the redundant tag column here rather than renaming over it.
    return games_to_df(run_games(jobs), label_a="AB", label_b="AB").drop(columns="tag")

dfB = run_or_load("expB_prealloc_sweep", expB)

# AB plays itself with sides alternated, so the X/O split is meaningless --
# what matters is how often games end decisively and how long they take.
summary_B = (dfB.groupby("n_prealloc")
                 .agg(decisiveness=("winner", lambda s: (s != "draw").mean()),
                      draw_rate   =("winner", lambda s: (s == "draw").mean()),
                      mean_plies  =("plies",  "mean"),
                      median_plies=("plies",  "median")))
summary_B

[run]   computing expB_prealloc_sweep.pkl ...
        done in 100.2s


,decisiveness,draw_rate,mean_plies,median_plies
n_prealloc,,,,
1,0.8875,0.1125,18.9750,17.5
2,0.8125,0.1875,16.8125,15.5
3,0.9375,0.0625,14.3500,13.0


## Experiment C -- heuristic-weight tuning

Grid-search around the default weights for the three knobs that matter most
(`w_three`, `w_block_three`, `w_center`) and play each candidate against the
default-weighted opponent at `TUNING_DEPTH`.  A `default vs default` control
row is included so we can verify the baseline win-rate is roughly 0.5 -- if
it is not, the comparison is biased and the rest of the table is suspect.

All ~400 games go into a single pool batch so the long candidates do not
block the short ones.

In [5]:
def expC():
    base = settings.DEFAULT_HEURISTIC_WEIGHTS
    grid = list(itertools.product(
        [base["w_three"]       * 0.5, base["w_three"],       base["w_three"]       * 2.0],
        [base["w_block_three"] * 0.5, base["w_block_three"], base["w_block_three"] * 2.0],
        [0.0,                          base["w_center"]       * 4.0],
    ))

    opp_spec = _spec("ab", depth=settings.TUNING_DEPTH)
    n        = settings.N_GAMES_PER_CONFIG

    # Map tag -> (weights, is_control) so we can split results out later.
    config_meta = {}
    jobs = []

    # Sanity-check row: default weights vs themselves should land near 50 %.
    config_meta["control"] = (dict(base), True)
    jobs.extend(make_balanced_jobs(opp_spec, opp_spec, 2, n,
                                   seed=300, tag="control"))

    # All grid candidates.
    for idx, (w_three, w_block_three, w_center) in enumerate(grid):
        weights = dict(base, w_three=w_three,
                       w_block_three=w_block_three, w_center=w_center)
        tag = f"cand_{idx:02d}"
        config_meta[tag] = (weights, False)
        cand_spec = _spec("ab", depth=settings.TUNING_DEPTH, weights=weights)
        jobs.extend(make_balanced_jobs(cand_spec, opp_spec, 2, n,
                                       seed=301 + idx, tag=tag))

    print(f"  expC submitting {len(jobs)} games to {MAX_WORKERS} workers ...")
    df_all = games_to_df(run_games(jobs), label_a="cand", label_b="base")

    rows = []
    for tag, (weights, is_control) in config_meta.items():
        sub = df_all[df_all["tag"] == tag]
        rows.append(dict(
            tag=tag, control=is_control,
            w_three      =weights["w_three"],
            w_block_three=weights["w_block_three"],
            w_center     =weights["w_center"],
            win_rate =(sub["winner_label"] == "cand").mean(),
            loss_rate=(sub["winner_label"] == "base").mean(),
            draw_rate=(sub["winner_label"] == "draw").mean(),
            mean_plies=sub["plies"].mean(),
        ))
    return pd.DataFrame(rows)

dfC = run_or_load("expC_weight_tuning", expC)
dfC.sort_values("win_rate", ascending=False)

[run]   computing expC_weight_tuning.pkl ...
  expC submitting 1520 games to 32 workers ...
        done in 540.1s


,tag,control,w_three,w_block_three,w_center,win_rate,loss_rate,draw_rate,mean_plies
15,cand_14,False,20.0,8.0,0.0,0.5750,0.4000,0.0250,15.2750
13,cand_12,False,20.0,4.0,0.0,0.5250,0.4500,0.0250,15.2000
14,cand_13,False,20.0,4.0,2.0,0.5125,0.4750,0.0125,14.7125
18,cand_17,False,20.0,16.0,2.0,0.5125,0.4125,0.0750,16.4875
11,cand_10,False,10.0,16.0,0.0,0.4875,0.4500,0.0625,16.4375
1,cand_00,False,5.0,4.0,0.0,0.4750,0.4625,0.0625,16.4750
10,cand_09,False,10.0,8.0,2.0,0.4625,0.4750,0.0625,15.0500
12,cand_11,False,10.0,16.0,2.0,0.4500,0.4500,0.1000,16.6500
2,cand_01,False,5.0,4.0,2.0,0.4500,0.4750,0.0750,16.8000
17,cand_16,False,20.0,16.0,0.0,0.4500,0.4875,0.0625,15.9500


### Validate the top candidates at the deployment depth

Tuning at `TUNING_DEPTH` is cheap but the winner there is not guaranteed to
still be the winner at `DEFAULT_DEPTH`, so we re-play the top-K candidates
at the deeper depth before committing to a set of weights.  If none of them
convincingly beats the baseline (>= 55 % win-rate), we keep the defaults.

In [6]:
candidates = dfC[~dfC["control"]].sort_values("win_rate", ascending=False)
ctrl_winrate = float(dfC[dfC["control"]].iloc[0]["win_rate"])
print(f"Control (default vs default) win-rate: {ctrl_winrate:.3f}  "
      f"(should be ~0.5; large deviations indicate a biased comparison)")

TOP_K        = 3
default_spec = _spec("ab", depth=settings.DEFAULT_DEPTH)
top          = candidates.head(TOP_K).reset_index(drop=True)

# Build all validation jobs up front so the pool stays saturated.
jobs            = []
weights_by_tag  = {}
for i, row in top.iterrows():
    weights = dict(settings.DEFAULT_HEURISTIC_WEIGHTS,
                   w_three      =float(row["w_three"]),
                   w_block_three=float(row["w_block_three"]),
                   w_center     =float(row["w_center"]))
    tag = f"val_{i}"
    weights_by_tag[tag] = weights
    cand_spec = _spec("ab", depth=settings.DEFAULT_DEPTH, weights=weights)
    jobs.extend(make_balanced_jobs(cand_spec, default_spec, 2,
                                   settings.N_GAMES_PER_CONFIG,
                                   seed=400 + i, tag=tag))

print(f"  validation submitting {len(jobs)} games ...")
df_v = games_to_df(run_games(jobs), label_a="cand", label_b="base")

validation = []
for i, row in top.iterrows():
    tag = f"val_{i}"
    sub = df_v[df_v["tag"] == tag]
    validation.append(dict(
        weights      =weights_by_tag[tag],
        tune_winrate =float(row["win_rate"]),
        d3_winrate   =(sub["winner_label"] == "cand").mean(),
        d3_lossrate  =(sub["winner_label"] == "base").mean(),
        d3_drawrate  =(sub["winner_label"] == "draw").mean()))
val_df = pd.DataFrame(validation)
print(val_df[["tune_winrate", "d3_winrate", "d3_lossrate", "d3_drawrate"]])

best_row = val_df.iloc[val_df["d3_winrate"].idxmax()]
if best_row["d3_winrate"] >= 0.55:
    best_weights = best_row["weights"]
    print(f"Selected tuned weights with d=3 win-rate {float(best_row['d3_winrate']):.3f}")
else:
    best_weights = dict(settings.DEFAULT_HEURISTIC_WEIGHTS)
    print("No tuned candidate beat the baseline at d=3; keeping default weights.")

settings.cache_path("best_weights.pkl").write_bytes(pickle.dumps(best_weights))
print("best weights saved to", settings.cache_path("best_weights.pkl"))
best_weights

Control (default vs default) win-rate: 0.412  (should be ~0.5; large deviations indicate a biased comparison)
  validation submitting 240 games ...
   tune_winrate  d3_winrate  d3_lossrate  d3_drawrate
0        0.5750      0.5500       0.3500       0.1000
1        0.5250      0.4625       0.4750       0.0625
2        0.5125      0.4375       0.4875       0.0750
Selected tuned weights with d=3 win-rate 0.550
best weights saved to C:\Git\AI801Project\cache\results\best_weights.pkl


{'w_two': 1.0,
 'w_three': 20.0,
 'w_block_two': 1.0,
 'w_block_three': 8.0,
 'w_center': 0.0,
 'w_win': 10000.0}